# SNEWPapers vs American Stories

Find an article through the snewpapers public API, rebuild its issue, and compare that issue with the same one in American Stories.

Your key is read from `.env` (see the README). Calls are cached under `data/snewpapers_api/`, so re-running a cell does not spend tokens again.

In [ ]:
from snewpapers_tools.api import SnewpapersClient

client = SnewpapersClient()
balance = client.tokens()
print(f"Balance: {balance['balance']} tokens "
      f"({balance['grant_remaining']} grant, {balance['purchased_remaining']} purchased)")

## 1. Find an article

Fill in the years, some keywords (any of them may match; articles matching more rank higher) and, if you like, a topic (one sentence, matched by meaning), then **Search**. One of the top hits is picked at random (the API has no random sort); **Pick another** until you like the article and its issue is a manageable size. **Compare this issue** confirms the estimate.

This notebook asks the API for caution articles (about 4% of the corpus: slurs, racist framing, graphic violence and the like) as printed, so the comparison covers the whole issue.

In [ ]:
from snewpapers_tools.form import ComparisonForm

form = ComparisonForm(client)
form.widget

## 2. Fetch both versions

After **Compare this issue**, run this cell. The first time for a year it downloads that year's American Stories archive (use the form's free download button, or it happens here) and reads through it once to copy out this issue's pages. Then it browses the whole issue through the API and fetches every component's text (at most the tokens shown above; texts you have already read are free). It stops before spending anything if American Stories lacks the issue. Everything is cached under `data/`.

In [ ]:
from snewpapers_tools.fetching import fetch_both

pair = fetch_both(client, form.confirmed)
print(f"{len(pair.components)} of {pair.total_components} API components "
      f"({len(pair.missing)} not returned), {len(pair.pages)} American Stories pages, "
      f"{pair.tokens_spent} tokens spent")
pair.per_page()

## 3. Page layouts

Both layouts drawn on the Library of Congress page scans, page columns aligned. Row 1 is American Stories: boxes coloured by their class (article, headline, author), tagged with the article they belong to (`a007`; `-` = in no article), with a dashed line through each article in reading order. Row 2 is snewpapers: each component's rectangles from the API (`include_boxes`, free) coloured by label family and tagged with its label. The scans download once per issue (about 10 MB a page) into `data/loc/`; the drawing is saved as WebP in `data/overlays/`.

In [ ]:
from IPython.display import display
from snewpapers_tools.overlay import components_by_page, issue_overlay

sheet, path = issue_overlay(pair)
print(path)
display(sheet)
components_by_page(pair.components)

## 4. Compare the segmentations

Free and local. Every snewpapers component is paired with the American Stories boxes inside it, and each pairing gets a case: the same article on both sides (`match`), one of ours holding several AS articles (`as_split`), several of ours inside one AS article (`as_merge`), AS text with none of ours (`as_only`), one of ours AS has no text for (`ours_only`), and so on. The README's "How to read the results" explains every case.

In [ ]:
from snewpapers_tools.segmentation import segment_issue
from snewpapers_tools.overlay import issue_scan_sizes

seg = segment_issue(pair, issue_scan_sizes(pair))
verdicts = None  # step 5 fills these in
seg.summary()

## 5. Ask an LLM where they differ (optional)

Needs `LLM_API_KEY`, `LLM_BASE_URL` and `LLM_MODEL` in `.env` (see the README). The model sees only the texts, never the scans or which system is which. A question averages about 1,000 input tokens (a long story up to about 10,000) and a short answer, plus whatever a reasoning model spends thinking; price that for your model. Answers are cached under `data/adjudications/`, so re-running costs nothing. Skip this cell to get the report without verdicts.

In [ ]:
from snewpapers_tools.adjudicate import Adjudicator, adjudicate, estimate_questions, scorecard

print(f"{estimate_questions(seg)} questions, about {estimate_questions(seg) * 1000:,} input tokens before the cache saves any")
adjudicator = Adjudicator()
try:
    verdicts = await adjudicate(seg, pair, adjudicator=adjudicator)
finally:
    await adjudicator.aclose()
scorecard(verdicts, seg)

## 6. The report and the verdict overlay

The report is a folder under `data/reports/` (open its `index.html` in a browser): a scorecard at the top, then every pairing with crops of both sides, the texts and, if you ran step 5, the verdict and the model's reasons. The overlay draws the verdicts on the page scans and is saved as WebP in `data/overlays/`.

In [ ]:
from snewpapers_tools.report import segmentation_report
from snewpapers_tools.verdict_overlay import verdict_overlay

_html, report_path = segmentation_report(pair, seg, verdicts=verdicts)
print(report_path)
sheet, overlay_path = verdict_overlay(pair, seg, verdicts)
print(overlay_path)
display(sheet)